# Feature extraction – visual data exploration

This notebook explores the frame-level CSV produced by `extract_features.py`. It is designed to help understand tracking quality, ball movement, player proximity, temporal signals and Smart Tagging **before tuning the possession prediction rules**.

**Input:** one row per video frame from `*_features.csv`.

## 1. Setup and load the feature CSV

Dependencies: `pandas`, `numpy`, `matplotlib`. Install if needed:

```bash
uv add pandas numpy matplotlib
```

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
CSV_PATH = Path("../data/derived/BAR-ATH_features_corrected.csv")

df = pd.read_csv(CSV_PATH)
df = df.sort_values("frame").reset_index(drop=True)

print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
display(df.head())

In [ ]:
import sys
from pathlib import Path

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from starter.baseline import HOME, AWAY, NOBODY

print("HOME:", HOME)
print("AWAY:", AWAY)
print("NOBODY:", NOBODY)

for col in ["tag_team_poss", "tag_team_def"]:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False).sort_index())

In [ ]:
# confirm pitch dimensions

from games import GAME, GAMES

pitch = GAMES[GAME]["pitch"]
length, width = pitch

print("Game:", GAME)
print("Pitch configuration:", pitch)
print(f"Length: {length} m")
print(f"Width:  {width} m")

## 2. Understand the table and data quality

Column types, missingness and value ranges. Missing values are meaningful here: for example, missing ball coordinates mean the ball was not tracked in that frame.

In [ ]:
display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_n": df.isna().sum(),
    "missing_pct": (100 * df.isna().mean()).round(2),
    "n_unique": df.nunique(dropna=True),
}).sort_values("missing_pct", ascending=False))

display(df.describe(include="all").T)

In [ ]:
# Missingness by feature (top 25)
missing_pct = 100 * df.isna().mean()
missing_pct = missing_pct[missing_pct > 0].sort_values()

if len(missing_pct):
    ax = missing_pct.tail(25).plot(kind="barh", figsize=(9, max(4, 0.28 * len(missing_pct.tail(25)))))
    ax.set(title="Missing values by feature (top 25)", xlabel="Missing frames (%)", ylabel="")
    ax.grid(axis="x", alpha=.25)
    plt.tight_layout()
    plt.show()
else:
    print("No pandas NaN values found.")

## 3. Frame coverage and tracking availability

Check whether all frames are represented, which frames belong to graded halves, and whether tracking availability changes over time. The expected output has one row per video frame.

In [ ]:
if "frame" in df:
    print("Frame sequence continuous:", np.array_equal(df["frame"].to_numpy(), np.arange(len(df))))
    print("First / last frame:", df["frame"].min(), "/", df["frame"].max())

if "half" in df:
    display(df["half"].value_counts(dropna=False).sort_index().rename("frames").to_frame())

if "graded" in df:
    display(df["graded"].value_counts(dropna=False).rename("frames").to_frame())

In [ ]:
# Plot tracking availability over time using time in seconds if FPS is known.
FPS = 25  # Update if this game uses a different frame rate.
x_time = df["frame"] / FPS if "frame" in df else np.arange(len(df)) / FPS

availability_cols = [c for c in ["n_home_tracked", "n_away_tracked"] if c in df]
if availability_cols:
    fig, ax = plt.subplots(figsize=(14, 4))
    for col in availability_cols:
        ax.plot(x_time, df[col], label=col, alpha=.8, linewidth=.7)
    ax.set(title="Tracked players per team over time", xlabel="Video time (s)", ylabel="Tracked players")
    ax.legend()
    ax.grid(alpha=.25)
    plt.tight_layout()
    plt.show()

if "ball_missing" in df:
    miss = df["ball_missing"].astype(bool)
    print(f"Ball missing: {miss.mean():.2%} of all frames")

## 4. Ball location on the pitch

A 2D density map shows where the ball is most often observed. It can reveal spatial coverage, long periods in one area, or coordinate problems. Coordinates are in metres according to the feature-extraction script.

This plot includes only observed ball positions; missing positions are excluded.

In [ ]:
if {"ball_x", "ball_y"}.issubset(df.columns):
    valid = df[["ball_x", "ball_y"]].dropna()
    fig, ax = plt.subplots(figsize=(11, 6))
    h = ax.hist2d(valid["ball_x"], valid["ball_y"], bins=45, cmap="viridis")
    plt.colorbar(h[3], ax=ax, label="Frame count")
    ax.set(title="Ball-position density", xlabel="Pitch length direction (m)", ylabel="Pitch width direction (m)")
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=.15)
    plt.tight_layout()
    plt.show()
    print(f"Frames with ball coordinates: {len(valid):,} ({len(valid)/len(df):.1%})")
else:
    print("ball_x / ball_y columns not available.")

## 5. Ball movement: speed and acceleration

Inspect the distribution and time series of ball speed. Very high spikes may be genuine kicks or tracking noise; compare them with the video before interpreting them. `ball_speed_smooth` is a rolling median in the extraction script.

In [ ]:
for col in ["ball_speed", "ball_speed_smooth", "ball_accel"]:
    if col in df:
        s = df[col].dropna()
        if len(s):
            fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
            axes[0].hist(s, bins=60)
            axes[0].set(title=f"{col}: distribution", xlabel=col, ylabel="Frame count")
            axes[1].plot(s.iloc[::max(1, len(s)//5000)].to_numpy(), linewidth=.6)
            axes[1].set(title=f"{col}: sampled sequence", xlabel="Sample index", ylabel=col)
            for ax in axes: ax.grid(alpha=.2)
            plt.tight_layout()
            plt.show()

In [ ]:
if {"frame", "ball_speed", "ball_speed_smooth"}.issubset(df.columns):
    # Plot a manageable segment; change START_S / DURATION_S to inspect a moment.
    START_S, DURATION_S = 0, 60
    start, end = int(START_S * FPS), int((START_S + DURATION_S) * FPS)
    seg = df.iloc[start:end]
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(seg["frame"]/FPS, seg["ball_speed"], alpha=.35, linewidth=.6, label="raw speed")
    ax.plot(seg["frame"]/FPS, seg["ball_speed_smooth"], linewidth=1, label="smoothed speed")
    ax.set(title=f"Ball speed – {START_S}s to {START_S + DURATION_S}s", xlabel="Time (s)", ylabel="Speed (m/s)")
    ax.legend(); ax.grid(alpha=.25)
    plt.tight_layout(); plt.show()

In [ ]:
if {"frame", "ball_speed", "ball_speed_smooth"}.issubset(df.columns):
    # Plot a manageable segment; change START_S / DURATION_S to inspect a moment.
    START_S2, DURATION_S2 = 40_462/FPS, 60
    start, end = int(START_S2 * FPS), int((START_S2 + DURATION_S2) * FPS)
    seg = df.iloc[start:end]
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(seg["frame"]/FPS, seg["ball_speed"], alpha=.35, linewidth=.6, label="raw speed")
    ax.plot(seg["frame"]/FPS, seg["ball_speed_smooth"], linewidth=1, label="smoothed speed")
    ax.set(title=f"Ball speed – {START_S2}s to {START_S2 + DURATION_S2}s", xlabel="Time (s)", ylabel="Speed (m/s)")
    ax.legend(); ax.grid(alpha=.25)
    plt.tight_layout(); plt.show()

In [ ]:
# quantify the coordinate problem

valid = df.loc[
    df["graded"] & df["ball_x"].notna() & df["ball_y"].notna()
].copy()

length, width = pitch

inside = (
    valid["ball_x"].between(0, length)
    & valid["ball_y"].between(0, width)
)

within_3m = (
    valid["ball_x"].between(-3, length + 3)
    & valid["ball_y"].between(-3, width + 3)
)

within_10m = (
    valid["ball_x"].between(-10, length + 10)
    & valid["ball_y"].between(-10, width + 10)
)

print(f"Valid ball observations: {len(valid):,}")
print(f"Inside pitch:             {inside.mean():.1%}")
print(f"Within 3m of pitch:       {within_3m.mean():.1%}")
print(f"Within 10m of pitch:      {within_10m.mean():.1%}")

print("\nCoordinate quantiles:")
display(valid[["ball_x", "ball_y"]].quantile(
    [0, .001, .01, .05, .5, .95, .99, .999, 1]
))

In [ ]:
# visualize the coordinates against the pitch

fig, ax = plt.subplots(figsize=(12, 7))

ax.scatter(
    valid["ball_x"],
    valid["ball_y"],
    s=2,
    alpha=0.15,
)

# Expected pitch boundaries
ax.plot(
    [0, length, length, 0, 0],
    [0, 0, width, width, 0],
    linewidth=2,
    label="Configured pitch boundary",
)

ax.set_xlim(-15, length + 15)
ax.set_ylim(-15, width + 15)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Ball X (m)")
ax.set_ylabel("Ball Y (m)")
ax.set_title("Ball tracking coordinates relative to pitch boundaries")
ax.legend()
ax.grid(alpha=0.2)

plt.show()

In [ ]:
def ball_motion(ball, fps):
    """Calculate ball speed and acceleration without bridging tracking gaps."""

    n = len(ball)

    valid_position = (
        np.isfinite(ball[:, 0])
        & np.isfinite(ball[:, 1])
    )

    # Only calculate speed between consecutive valid observations.
    valid_step = valid_position[1:] & valid_position[:-1]

    speed = np.full(n, np.nan, dtype=np.float32)

    distances = np.hypot(
        np.diff(ball[:, 0]),
        np.diff(ball[:, 1]),
    )

    speed[1:][valid_step] = distances[valid_step] * fps

    # Smooth independently within contiguous valid speed segments.
    smooth_speed = np.full(n, np.nan, dtype=np.float32)

    valid_indices = np.flatnonzero(np.isfinite(speed))

    if len(valid_indices):
        split_positions = np.where(
            np.diff(valid_indices) > 1
        )[0] + 1

        segments = np.split(valid_indices, split_positions)

        for indices in segments:
            segment_speed = pd.Series(speed[indices])

            smooth_speed[indices] = (
                segment_speed
                .rolling(
                    SPEED_SMOOTH_FRAMES,
                    center=True,
                    min_periods=1,
                )
                .median()
                .to_numpy()
            )

    # Acceleration only between consecutive valid smoothed speeds.
    acceleration = np.full(n, np.nan, dtype=np.float32)

    valid_accel = (
        np.isfinite(smooth_speed[1:])
        & np.isfinite(smooth_speed[:-1])
    )

    acceleration[1:][valid_accel] = (
        np.diff(smooth_speed)[valid_accel] * fps
    )

    return speed, smooth_speed, acceleration

In [ ]:
print(df[["ball_speed", "ball_speed_smooth", "ball_accel"]].describe(
    percentiles=[.5, .75, .90, .95, .99, .999]
))

In [ ]:
for col in ["ball_speed", "ball_speed_smooth", "ball_accel"]:
    print(col, df[col].notna().mean())

## 6. Player proximity and the nearest-team signal

`d_home` and `d_away` are the distances from the ball to the nearest tracked player on each team. `margin = d_away - d_home`: positive values mean the nearest HOME player is closer; negative values mean the nearest AWAY player is closer.

This is a proximity signal, **not proof of possession**.

In [ ]:
if {"d_home", "d_away"}.issubset(df.columns):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for col in ["d_home", "d_away"]:
        if col in df:
            axes[0].hist(df[col].dropna(), bins=50, alpha=.55, label=col)
    axes[0].set(title="Nearest-player distance distributions", xlabel="Distance to ball (m)", ylabel="Frame count")
    axes[0].legend(); axes[0].grid(alpha=.2)

    if "margin" in df:
        axes[1].hist(df["margin"].dropna(), bins=60)
        axes[1].axvline(0, linestyle="--")
        axes[1].set(title="Distance margin distribution", xlabel="d_away - d_home (m)", ylabel="Frame count")
        axes[1].grid(alpha=.2)
    plt.tight_layout(); plt.show()

In [ ]:
# Time series of nearest distances and margin for a selected window.
if {"frame", "d_home", "d_away"}.issubset(df.columns):
    START_S, DURATION_S = 0, 60
    seg = df.iloc[int(START_S*FPS):int((START_S+DURATION_S)*FPS)]
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    axes[0].plot(seg["frame"]/FPS, seg["d_home"], label="HOME nearest", linewidth=.8)
    axes[0].plot(seg["frame"]/FPS, seg["d_away"], label="AWAY nearest", linewidth=.8)
    axes[0].set(ylabel="Distance (m)", title="Nearest player distance")
    axes[0].legend(); axes[0].grid(alpha=.2)
    if "margin" in seg:
        axes[1].plot(seg["frame"]/FPS, seg["margin"], linewidth=.8)
        axes[1].axhline(0, color="black", linestyle="--", linewidth=.8)
        axes[1].set(ylabel="Margin (m)", xlabel="Time (s)", title="Distance margin (positive = HOME closer)")
        axes[1].grid(alpha=.2)
    plt.tight_layout(); plt.show()

## 7. Number of players near the ball

The extraction code uses a 5 m `near` radius and a 2 m `very-near` radius. Compare teams and inspect how often there are no players nearby, one nearby player, or several players crowding the ball.

In [ ]:
count_cols = [c for c in ["n_home_near", "n_away_near", "n_home_very_near", "n_away_very_near"] if c in df]
if count_cols:
    fig, ax = plt.subplots(figsize=(10, 4))
    df[count_cols].mean().plot(kind="bar", ax=ax)
    ax.set(title="Average number of players near the ball", ylabel="Mean players per frame", xlabel="")
    ax.grid(axis="y", alpha=.25)
    plt.tight_layout(); plt.show()

    display(df[count_cols].describe().T)

In [ ]:
if {"n_home_near", "n_away_near"}.issubset(df.columns):
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(df["n_home_near"], df["n_away_near"], alpha=.12, s=8)
    ax.set(title="HOME vs AWAY players within 5 m", xlabel="HOME count", ylabel="AWAY count")
    ax.grid(alpha=.2)
    plt.tight_layout(); plt.show()

## 8. Nearest team, persistence and run lengths

`nearest_team` encodes the team whose player is closest to the ball (the numeric codes come from the baseline implementation). The share features are trailing proportions over 25 and 50 frames; `nearest_team_run_frames` counts consecutive frames with the same nearest team.

Use the value counts and run-length distributions to see whether the raw nearest-team signal flickers frequently.

In [ ]:
if "nearest_team" in df:
    display(df["nearest_team"].value_counts(dropna=False).sort_index().rename("frames").to_frame())
    print("Check the HOME / AWAY / NOBODY numeric mapping in starter.baseline before interpreting codes.")

share_cols = [c for c in df.columns if c.startswith("nearest_home_share_") or c.startswith("nearest_away_share_")]
if share_cols:
    fig, ax = plt.subplots(figsize=(14, 4))
    sample_step = max(1, len(df)//8000)
    for col in share_cols:
        ax.plot(x_time[::sample_step], df[col].iloc[::sample_step], label=col, linewidth=.8)
    ax.set(title="Trailing nearest-team shares", xlabel="Time (s)", ylabel="Share")
    ax.legend(ncol=2); ax.grid(alpha=.2)
    plt.tight_layout(); plt.show()

if "nearest_team_run_frames" in df:
    runs = df.loc[df["nearest_team_run_frames"].diff().fillna(1) <= 0, "nearest_team_run_frames"]
    # More reliable run extraction: each new run starts when run length returns to 1.
    starts = df["nearest_team_run_frames"].eq(1)
    run_lengths = df.loc[starts, "nearest_team_run_frames"]
    # The run start row is 1, so derive completed run lengths from resets.
    vals = df["nearest_team_run_frames"].to_numpy()
    ends = np.r_[np.flatnonzero(np.diff(vals) < 0), len(vals)-1]
    completed = vals[ends] if len(vals) else np.array([])
    if len(completed):
        plt.figure(figsize=(9, 3.5))
        plt.hist(completed / FPS, bins=50)
        plt.xlabel("Consecutive nearest-team run duration (s)")
        plt.ylabel("Number of runs")
        plt.title("Nearest-team run duration distribution")
        plt.grid(alpha=.2); plt.tight_layout(); plt.show()

## 9. Smart Tagging coverage and overlaps

Inspect the frequency and duration of tags. These annotations may be sparse, and their frame ranges can overlap. The plots show tag presence, not whether the annotations are correct.

In [ ]:
tag_cols = [c for c in df.columns if c.startswith("tag_") and df[c].dtype != "object"]

rows = []
if tag_cols:
    for c in tag_cols:
        if c in ["tag_team_poss", "tag_team_def"]:
            for value, label in [
                (HOME, "HOME"),
                (AWAY, "AWAY"),
                (NOBODY, "NOBODY"),
            ]:
                active = (df[c] == value)
                rows.append({
                    "tag_column": c,
                    "label": label,
                    "active_frames": int(active.sum()),
                    "active_pct": 100 * active.mean(),
                })
        else:
            active = df[c].fillna(False).astype(bool)
            rows.append({
                "tag_column": c,
                "label": "active",
                "active_frames": int(active.sum()),
                "active_pct": 100 * active.mean(),
            })

    tag_summary = pd.DataFrame(rows)
    display(tag_summary.sort_values(["tag_column", "label"]))

    # Plot only the boolean-style tags, since team-coded columns are reported by team value.
    boolean_rows = tag_summary[tag_summary["label"] == "active"]
    if not boolean_rows.empty:
        ax = boolean_rows.set_index("tag_column")["active_pct"].sort_values().plot(kind="barh", figsize=(8, 4))
        ax.set(title="Smart Tagging coverage", xlabel="Frames active (%)", ylabel="")
        ax.grid(axis="x", alpha=.25)
        plt.tight_layout()
        plt.show()

if "tag_set_piece_type" in df:
    display(df.loc[df["tag_set_piece_type"].fillna("").ne(""), "tag_set_piece_type"].value_counts().to_frame("frames"))


In [ ]:
# Plot tag intervals in a selected time window as a compact raster.
raster_cols = [c for c in ["tag_dead", "tag_goal"] if c in df]
for c in ["tag_team_poss", "tag_team_def"]:
    if c in df: raster_cols.append(c)

if raster_cols:
    START_S, DURATION_S = 0, 120
    seg = df.iloc[int(START_S*FPS):int((START_S+DURATION_S)*FPS)]
    fig, ax = plt.subplots(figsize=(14, max(2.5, .45*len(raster_cols))))
    for row, col in enumerate(raster_cols):
        values = seg[col]
        if col in ["tag_dead", "tag_goal"]:
            active = values.fillna(False).astype(bool).to_numpy()
        else:
            # Numeric team codes: plot non-zero values; consult baseline mapping.
            active = values.fillna(0).to_numpy() != 0
        ax.scatter(seg.loc[active, "frame"]/FPS, np.full(active.sum(), row), s=3, label=col)
    ax.set_yticks(range(len(raster_cols)), raster_cols)
    ax.set(xlabel="Time (s)", title="Smart Tagging activity timeline")
    ax.grid(axis="x", alpha=.2); plt.tight_layout(); plt.show()

In [ ]:
# Smart Tagging timeline explicit team-code handling

raster_cols = ["tag_dead", "tag_goal", "tag_team_poss", "tag_team_def"]
raster_cols = [c for c in raster_cols if c in df.columns]

START_S = 1618
DURATION_S = 60

seg = df.iloc[int(START_S * FPS):int((START_S + DURATION_S) * FPS)]

fig, ax = plt.subplots(figsize=(15, 5))

for row, col in enumerate(raster_cols):
    values = seg[col]

    if col in ["tag_dead", "tag_goal"]:
        active = values.fillna(False).astype(bool).to_numpy()
        ax.scatter(
            seg.loc[active, "frame"] / FPS,
            np.full(active.sum(), row),
            s=5,
            label=col,
        )
    else:
        home = (values == HOME).to_numpy()
        away = (values == AWAY).to_numpy()
        unknown = (values == NOBODY).to_numpy()

        ax.scatter(
            seg.loc[home, "frame"] / FPS,
            np.full(home.sum(), row),
            s=5,
            label=f"{col} — HOME",
        )
        ax.scatter(
            seg.loc[away, "frame"] / FPS,
            np.full(away.sum(), row),
            s=5,
            label=f"{col} — AWAY",
        )
        # Unknown values are deliberately not plotted as active tags.

ax.set_yticks(range(len(raster_cols)), raster_cols)
ax.set_xlabel("Time (s)")
ax.set_title("Smart Tagging activity — explicit team assignments")
ax.grid(axis="x", alpha=0.2)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 10. Compare halves / graded and ungraded sections

Use this to spot differences in tracking coverage or feature distributions between halves. Do not interpret ungraded sections as labelled possession data.

In [ ]:
if "half" in df:
    numeric_candidates = [c for c in [
        "ball_speed_smooth", "d_home", "d_away", "margin",
        "n_home_near", "n_away_near", "n_home_tracked", "n_away_tracked"
    ] if c in df]
    if numeric_candidates:
        display(df.groupby("half")[numeric_candidates].agg(["mean", "median", "std"]).round(2))

## 11. Feature relationships

A correlation heatmap can show redundant signals or unexpected relationships. Correlation is descriptive, not causal, and binary/team-coded columns should be interpreted carefully.

In [ ]:
corr_cols = [c for c in [
    "ball_speed_smooth", "ball_accel", "d_home", "d_away", "margin",
    "n_home_near", "n_away_near", "n_home_very_near", "n_away_very_near",
    "nearest_home_share_25f", "nearest_away_share_25f",
    "nearest_home_share_50f", "nearest_away_share_50f",
    "d_home_change_10f", "d_away_change_10f"
] if c in df.columns]
if len(corr_cols) >= 2:
    corr = df[corr_cols].corr()
    fig, ax = plt.subplots(figsize=(11, 8))
    im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm", aspect="auto")
    ax.set_xticks(range(len(corr_cols)), corr_cols, rotation=90)
    ax.set_yticks(range(len(corr_cols)), corr_cols)
    fig.colorbar(im, ax=ax, label="Pearson correlation")
    ax.set_title("Feature correlation matrix")
    plt.tight_layout(); plt.show()

## 12. Build a compact diagnostic table

This table highlights candidate data-quality issues for manual inspection. Thresholds are prompts for investigation, not universal definitions of bad data.

In [ ]:
diagnostics = {}
if "ball_missing" in df:
    diagnostics["ball_missing_pct"] = 100 * df["ball_missing"].fillna(False).mean()
if "ball_out" in df:
    diagnostics["ball_out_pct"] = 100 * df["ball_out"].fillna(False).mean()
for c in ["n_home_tracked", "n_away_tracked"]:
    if c in df:
        diagnostics[f"{c}_zero_pct"] = 100 * (df[c].fillna(0) == 0).mean()
for c in ["d_home", "d_away", "ball_speed_smooth"]:
    if c in df:
        diagnostics[f"{c}_missing_pct"] = 100 * df[c].isna().mean()

display(pd.Series(diagnostics, name="value").to_frame())

In [ ]:
# coordinates diagnostics

valid = df.loc[
    df["graded"]
    & df["ball_x"].notna()
    & df["ball_y"].notna()
].copy()

length, width = 105.3, 68.0

inside = (
    valid["ball_x"].between(0, length)
    & valid["ball_y"].between(0, width)
)

within_3m = (
    valid["ball_x"].between(-3, length + 3)
    & valid["ball_y"].between(-3, width + 3)
)

within_10m = (
    valid["ball_x"].between(-10, length + 10)
    & valid["ball_y"].between(-10, width + 10)
)

print(f"Valid observations: {len(valid):,}")
print(f"Inside pitch:       {inside.mean():.2%}")
print(f"Within 3m:          {within_3m.mean():.2%}")
print(f"Within 10m:         {within_10m.mean():.2%}")

print("\nCoordinate quantiles:")
display(
    valid[["ball_x", "ball_y"]].quantile(
        [0, .001, .01, .05, .25, .50, .75, .95, .99, .999, 1]
    )
)

## Ball tracking and direction of travel along x

In [ ]:
# Ball tracking coordinates relative to pitch boundaries, colored by direction of travel along x.
# If a prediction CSV is available, we overlay a sparse sample of labels for context.

pred_path = Path("../evaluation/my_results/BAR-ATH_predictions.csv")
if pred_path.exists():
    pred = pd.read_csv(pred_path)
else:
    pred = None

plot_df = df[["frame", "ball_x", "ball_y"]].copy()
if pred is not None:
    plot_df = plot_df.merge(pred, on="frame", how="left")

valid = plot_df.dropna(subset=["ball_x", "ball_y"]).copy()
valid["dx"] = valid["ball_x"].diff().fillna(0)
valid["x_direction"] = np.where(valid["dx"] >= 0, "toward x=120", "toward x=0")

plot_colors = {
    "toward x=0": "#4c78a8",
    "toward x=120": "#f58518",
}
pred_colors = {"H": "#2ca02c", "A": "#d62728", "D": "#7f7f7f", "X": "#b0b0b0"}

fig, ax = plt.subplots(figsize=(12, 7))

for direction, color in plot_colors.items():
    group = valid[valid["x_direction"] == direction]
    ax.scatter(group["ball_x"], group["ball_y"], s=10, alpha=0.35, color=color, label=direction)

# Expected pitch boundaries.
ax.plot([0, length, length, 0, 0], [0, 0, width, width, 0], color="black", linewidth=2, label="pitch boundary")

# Overlay a sparse subset of prediction labels if available.
if pred is not None and "label" in valid.columns:
    labeled = valid.dropna(subset=["label"]).copy()
    if len(labeled):
        sample_step = max(1, len(labeled) // 150)
        sample = labeled.iloc[::sample_step]

        for lab, color in pred_colors.items():
            subset = sample[sample["label"] == lab]
            if len(subset):
                ax.scatter(
                    subset["ball_x"], subset["ball_y"],
                    s=28, marker="o", edgecolors="black", linewidths=0.5,
                    facecolors=color, alpha=0.8, label=f"pred {lab}"
                )

        text_sample = sample.iloc[::max(1, len(sample) // 25)]
        for _, row in text_sample.iterrows():
            ax.text(row["ball_x"], row["ball_y"], row["label"], fontsize=8, color="black", alpha=0.8)

ax.set_xlim(-10, length + 10)
ax.set_ylim(-10, width + 10)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Ball X (m)")
ax.set_ylabel("Ball Y (m)")
ax.set_title("Ball tracking coordinates relative to pitch boundaries\ncolored by progression toward x=0 vs x=120")
ax.grid(alpha=0.2)
ax.legend(loc="best", framealpha=0.8)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from pathlib import Path

PRED_COLORS = {"H": "#2ca02c", "A": "#d62728", "D": "#7f7f7f", "X": "#b0b0b0"}
DIR_COLORS = {"toward_0": "#4c78a8", "toward_L": "#f58518"}


# ---------- data prep ----------
def prepare_trajectory(df, pred=None, smooth_win=25, max_jump=10.0, max_frame_gap=1):
    """Sort, merge predictions, smooth direction, and tag continuous segments."""
    out = df[["frame", "ball_x", "ball_y"]].copy()
    if pred is not None:
        out = out.merge(pred, on="frame", how="left")
    out = out.dropna(subset=["ball_x", "ball_y"]).sort_values("frame").reset_index(drop=True)

    # Smoothed x-velocity -> stable direction (no per-frame flicker)
    out["dx"] = out["ball_x"].diff().fillna(0)
    out["dx_smooth"] = out["dx"].rolling(smooth_win, center=True, min_periods=1).mean()

    # Break the line on frame gaps or implausible jumps (m between consecutive frames)
    step = np.hypot(out["ball_x"].diff(), out["ball_y"].diff())
    gap = out["frame"].diff()
    out["segment"] = ((step > max_jump) | (gap > max_frame_gap)).cumsum()
    return out


# ---------- drawing helpers ----------
def draw_pitch(ax, length, width, lw=1.5):
    ax.plot([0, length, length, 0, 0], [0, 0, width, width, 0], color="black", lw=lw, zorder=5)
    ax.plot([length / 2] * 2, [0, width], color="black", lw=lw * 0.6, alpha=0.6, zorder=5)
    ax.set_xlim(-5, length + 5)
    ax.set_ylim(-5, width + 5)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(alpha=0.15)


def add_trajectory(ax, data, length, color_by="direction", lw=0.8, alpha=0.7, cmap="viridis"):
    """Draw trajectory as thin line segments, never connecting across breaks."""
    pts = data[["ball_x", "ball_y"]].to_numpy()
    if len(pts) < 2:
        return None
    segs = np.stack([pts[:-1], pts[1:]], axis=1)
    seg_ids = data["segment"].to_numpy()
    keep = seg_ids[:-1] == seg_ids[1:]
    segs = segs[keep]

    if color_by == "direction":
        toward_L = (data["dx_smooth"].to_numpy()[:-1] >= 0)[keep]
        colors = np.where(toward_L, DIR_COLORS["toward_L"], DIR_COLORS["toward_0"])
        lc = LineCollection(segs, colors=colors, linewidths=lw, alpha=alpha, capstyle="round")
    else:  # color by time
        t = data["frame"].to_numpy()[:-1][keep]
        lc = LineCollection(segs, cmap=cmap, linewidths=lw, alpha=alpha, capstyle="round")
        lc.set_array(t)
        lc.set_clim(data["frame"].min(), data["frame"].max())
    ax.add_collection(lc)
    return lc


def add_predictions(ax, data, max_markers=60, label_every=3):
    """Sparse overlay of predicted labels, drawn on top of the lines."""
    if "label" not in data.columns:
        return
    labeled = data.dropna(subset=["label"])
    if labeled.empty:
        return
    sample = labeled.iloc[:: max(1, len(labeled) // max_markers)]
    for lab, color in PRED_COLORS.items():
        sub = sample[sample["label"] == lab]
        if len(sub):
            ax.scatter(sub["ball_x"], sub["ball_y"], s=22, c=color, edgecolors="black",
                       linewidths=0.5, zorder=6)
    for _, r in sample.iloc[::label_every].iterrows():
        ax.annotate(r["label"], (r["ball_x"], r["ball_y"]), xytext=(3, 3),
                    textcoords="offset points", fontsize=7, zorder=7)


def legend_handles(length, color_by, show_pred):
    h = []
    if color_by == "direction":
        h += [Line2D([], [], color=DIR_COLORS["toward_0"], lw=2, label="toward x=0"),
              Line2D([], [], color=DIR_COLORS["toward_L"], lw=2, label=f"toward x={length:g}")]
    if show_pred:
        h += [Line2D([], [], marker="o", ls="", mfc=c, mec="black", label=f"pred {k}")
              for k, c in PRED_COLORS.items()]
    return h


# ---------- main plots ----------
def plot_overview(data, length, width, color_by="time", show_pred=True, lw=0.6, alpha=0.6):
    fig, ax = plt.subplots(figsize=(12, 7.5))
    lc = add_trajectory(ax, data, length, color_by=color_by, lw=lw, alpha=alpha)
    draw_pitch(ax, length, width)
    if show_pred:
        add_predictions(ax, data)
    if color_by == "time" and lc is not None:
        fig.colorbar(lc, ax=ax, fraction=0.025, pad=0.02, label="frame")
    ax.legend(handles=legend_handles(length, color_by, show_pred), loc="upper left",
              bbox_to_anchor=(1.01, 1) if color_by == "direction" else None, framealpha=0.9)
    ax.set(xlabel="Ball X (m)", ylabel="Ball Y (m)", title="Ball trajectory")
    plt.tight_layout()
    plt.show()


def plot_windows(data, length, width, n_windows=6, color_by="direction", ncols=3,
                 show_pred=True, lw=1.0, alpha=0.8):
    """Small multiples: one panel per time window -> trajectories stay readable."""
    bins = pd.qcut(data["frame"], n_windows, labels=False, duplicates="drop")
    nrows = int(np.ceil(bins.nunique() / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 3.8 * nrows), squeeze=False)
    for ax, (b, chunk) in zip(axes.ravel(), data.groupby(bins)):
        add_trajectory(ax, chunk, length, color_by=color_by, lw=lw, alpha=alpha)
        draw_pitch(ax, length, width)
        if show_pred:
            add_predictions(ax, chunk, max_markers=15, label_every=1)
        ax.set_title(f"frames {chunk['frame'].min()}–{chunk['frame'].max()}", fontsize=9)
        ax.tick_params(labelsize=7)
    for ax in axes.ravel()[bins.nunique():]:
        ax.axis("off")
    fig.legend(handles=legend_handles(length, color_by, show_pred), loc="lower center",
               ncol=6, frameon=False)
    fig.suptitle("Ball trajectory by time window", y=0.995)
    plt.tight_layout(rect=(0, 0.04, 1, 1))
    plt.show()


# ---------- usage ----------
pred_path = Path("../evaluation/my_results/BAR-ATH_predictions.csv")
pred = pd.read_csv(pred_path) if pred_path.exists() else None

traj = prepare_trajectory(df, pred, smooth_win=25, max_jump=10.0)

plot_overview(traj, length, width, color_by="time")                 # whole match, colored by time
plot_windows(traj, length, width, n_windows=6, color_by="direction")  # readable per-period view